In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

from tqdm import tqdm
import torch
import torch.nn.functional as F
import numpy as np

from encoder import ConvEncoder
from contextualizer import Contextualizer
from BENDRTrainer import BENDRTrainer
from dataset import EEGDataset
from transforms import RandomTemporalCrop

In [3]:
pretrain_args = {
  "mask_rate": 0.065,
  "mask_span": 10,
  "layer_drop": 0.01,
  "multi_gpu": True,
  "temp": 0.1,
  "encoder_grad_frac": 0.1,
  "num_negatives": 20,
  "enc_feat_l2": 1.0
}

In [4]:
encoder = ConvEncoder(19, encoder_h=512)
contextualizer = Contextualizer(encoder.encoder_h, layer_drop=0.01)
trainer = BENDRTrainer(encoder, contextualizer, **pretrain_args)

No GPU detected: training and model execution will be performed on CPU.


In [5]:
# Slower learning rate for the encoder
trainer.set_optimizer(torch.optim.Adam(trainer.parameters()))
trainer.add_batch_transform(RandomTemporalCrop(max_crop_frac=0.05))

In [6]:
def load_dataset(path):
    ecs = []
    eos = []

    for subject in tqdm(os.listdir(path)):
        ec = torch.load(os.path.join(path, subject, 'EC.pt'), weights_only=True)
        eo = torch.load(os.path.join(path, subject, 'EO.pt'), weights_only=True)

        ecs.append(ec)
        eos.append(eo)

    ecs = torch.cat(ecs, axis=0).float()
    eos = torch.cat(eos, axis=0).float()

    dataset = torch.cat([ecs, eos], axis=0)
    print("Dataset shape:", dataset.shape)

    return dataset

In [7]:
dataset = load_dataset('/home/azureuser/mycontainer/SmallSubset')
print(dataset[0][0].shape, dataset[0][0].dtype)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:03<00:00,  6.13it/s]

Dataset shape: torch.Size([80, 19, 15360])
torch.Size([15360]) torch.float32


In [8]:
trainer.fit(training_dataset=dataset, epochs=5, batch_size=8)

Loading data with 4 additional workers


Epoch 0:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  | 0/10 [00:00<?, ?it/s]

/anaconda/envs/EEG-ML/lib/python3.8/site-packages/torch/nn/modules/module.py:1640: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
Epoch 0:  10%|████████████████████████████████████████████████████████████████████████████████████████████▌                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

torch.Size([8, 19, 15360]) torch.Size([8, 512, 160])


Epoch 0:  20%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    | 2/10 [00:12<00:50,  6.27s/it, Similarity=2.16, loss=5.94]

...

torch.Size([8, 19, 15360]) torch.Size([8, 512, 160])


Epoch 2:  80%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                                                                                                                                                                         | 8/10 [00:47<00:11,  5.74s/it, Similarity=4.29, loss=2.27]

torch.Size([8, 19, 15360]) torch.Size([8, 512, 160])


Epoch 2:  80%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                                                                                                                                                                         | 8/10 [00:48<00:12,  6.06s/it, Similarity=4.29, loss=2.27]


KeyboardInterrupt: 